# Gym 

In this assignment, you will use the `gym.csv` dataset to predict whether a member has renewed their membership.    
***Notes:***

- Some parts of the code are already provided. **Do not modify the existing code.**
- Write your solution only in the sections marked with `### YOUR SOLUTION`.
- You can verify automatically graded tasks using the cell labeled `### TEST` after each function.

***IMPORTANT NOTE:***
- Name your Jupyter Notebook as `gym_{index}.ipynb`.
- For example, if your index is 123456, you should name your notebook as `gym_12346.ipynb`.

In [1]:
import os
import hashlib
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import (
    PolynomialFeatures,
    StandardScaler,
    MinMaxScaler,
    LabelEncoder,
    OrdinalEncoder,
)
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)
from sklearn.linear_model import (
    LinearRegression,
    Lasso,
    Ridge,
    LassoCV,
    RidgeCV,
    LogisticRegression,
)
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor

In [2]:
RANDOM_STATE=42

In [3]:
def hash_data_frame(df):
    df_sorted = df.sort_index(axis=1).sort_values(by=list(df.columns))
    return hashlib.sha256(pd.util.hash_pandas_object(df_sorted, index=True).values).hexdigest()

In [4]:
def hash_series(series):
    series_str = ",".join(map(str, series.values))
    return hashlib.sha256(series_str.encode()).hexdigest()

In [5]:
def check_signature(expected, actual):
    # print(actual)
    try:
        assert actual == expected
    except AssertionError:
        raise

In [6]:
def test_func(func, signature):
    df = pd.read_csv("gym.csv")
    df = func(df)
    check_signature(signature, hash_data_frame(df))

In [7]:
def test_partition(func, train_X_signature, test_X_signature, train_y_signature, test_y_signature):
    df = pd.read_csv("gym.csv")
    train_X, test_X, train_y, test_y = func(df)
    try:
        # print(hash_data_frame(train_X))
        # print(hash_data_frame(test_X))
        # print(hash_series(train_y))
        # print(hash_series(test_y))
        assert hash_data_frame(train_X) == train_X_signature
        assert hash_data_frame(test_X) == test_X_signature
        assert hash_series(train_y) == train_y_signature
        assert hash_series(test_y) == test_y_signature
    except AssertionError:
        raise

In [8]:
df = pd.read_csv("gym.csv")

df.sample()

,member_id,city,age,gender,membership_duration_months,membership_type,monthly_visits,avg_session_minutes,monthly_active_minutes,bmi,membership_renewed
1430,CUST-001430,Prilep,23.0,Male,8.0,Monthly,NaN,NaN,241.0,Normal,0


In [9]:
### AUTOMATICALLY GRADED TASK
def calculate_descriptive_statistics(df):
    """
    Calculate the descriptive statistics for all numeric variables in the dataset.
    The statistics should include: count, mean, standard deviation, minimum,
    25th percentile, median, 75th percentile, and maximum.
    
    Return the result as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    return df.describe()
    ### END SOLUTION

In [10]:
statistics = calculate_descriptive_statistics(df)

In [11]:
### TEST
assert isinstance(statistics, pd.DataFrame)
required_stats = ["count", "mean", "std", "min", "25%", "50%", "75%", "max"]
for stat in required_stats:
    assert stat in statistics.index
numeric_columns = df.select_dtypes(include="number").columns
assert list(statistics.columns) == list(numeric_columns)

In [12]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_member_id(df):
    """
    Encode the `member_id` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    return df.drop(columns=["member_id"])    
    ### END SOLUTION

In [13]:
df = encode_or_drop_member_id(df)

In [14]:
### TEST
test_func(encode_or_drop_member_id, "8ca212a2a50076f1fbca7e9b9ee215cf2e9a6dbeb2d645b6096524cabfc4ac48")

In [15]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_city(df):
    """
    Encode the `city` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
        
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    return df.drop(columns=["city"])    
    ### END SOLUTION

In [16]:
df = encode_or_drop_city(df)

In [17]:
### TEST
test_func(encode_or_drop_city, "4b2aad1703efe3f45419dd1c9c0b733458db2b701275ad04595cd8455f5c17b3")

In [18]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_gender(df):
    """
    Encode the `gender` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    gender_dummies = pd.get_dummies(df["gender"], prefix="gender")
    df = pd.concat([df.drop(columns=["gender"]), gender_dummies], axis=1)
    return df
    ### END SOLUTION

In [19]:
df = encode_or_drop_gender(df)

In [20]:
### TEST
test_func(encode_or_drop_gender, "7446e30ca09384f0eb098be2cce623044725de7131bcc2283c98fbb2623c1abb")

In [21]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_membership_type(df):
    """
    Encode the `membership_type` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.

    If you encode `membership_type`, make sure the encoding reflects the exact duration in months as floats.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    mapping = {
        "Monthly": 1.0,
        "Quarterly": 3.0,
        "Annually": 12.0
    }

    df["membership_type"] = df["membership_type"].map(mapping)
    return df
    ### END SOLUTION

In [22]:
df = encode_or_drop_membership_type(df)

In [23]:
### TEST
test_func(encode_or_drop_membership_type, "3af0906d176ce91297ca309793bfef8c4595d1cf0b57e91db85252cfd3104ad0")

In [24]:
### AUTOMATICALLY GRADED TASK
def encode_or_drop_bmi(df):
    """
    Encode the `bmi` variable or remove it from the dataset.

    Note: If you plan to perform one-hot encoding, use the `pd.get_dummies` function, 
    use the original column name as a prefix for the new columns and
    append the new columns to the dataset. Also, remove the original column.
    
    Return the dataset as `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    encoder = OrdinalEncoder(
        categories=[["Underweight", "Normal", "Overweight", "Obese", np.nan]]
    )
    df[["bmi"]] = encoder.fit_transform(df[["bmi"]])
    return df
    ### END SOLUTION

In [25]:
df = encode_or_drop_bmi(df)

In [26]:
### TEST
test_func(encode_or_drop_bmi, "cd1a270ee12212b1a6d7d6c040386f7659c2c30a61fa39efaaf61bb5b0658279")

In [27]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_age(df):
    """
    Impute or remove the missing values from the `age` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df['age'] = df['age'].fillna(df['age'].median())
    return df
    ### END SOLUTION

In [28]:
df = handle_missing_values_in_age(df)

In [29]:
### TEST
test_func(handle_missing_values_in_age, "dc6e8fb4d44206bf89cc6b9c38caccaab6c924faf3aa5726015c94138cd3eaa5")

In [30]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_membership_duration_months(df):
    """
    Impute or remove the missing values from the `membership_duration_months` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df['membership_duration_months'] = df['membership_duration_months'].fillna(df['membership_duration_months'].median())
    return df
    ### END SOLUTION

In [31]:
df = handle_missing_values_in_membership_duration_months(df)

In [32]:
### TEST
test_func(handle_missing_values_in_membership_duration_months, "5603d56d498b19a4d764b78a1333e4ac393676c51e41f343b51fad2c097075c1")

In [33]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_monthly_visits_and_monthly_active_minutes(df):
    """
    Impute or remove the missing values from the `monthly_visits and monthly_active_minutes` columns.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    imputer = IterativeImputer(random_state=42)

    df[["monthly_visits", "monthly_active_minutes"]] = imputer.fit_transform(
        df[["monthly_visits", "monthly_active_minutes"]]
    )    
    return df
    ### END SOLUTION

In [34]:
df = handle_missing_values_in_monthly_visits_and_monthly_active_minutes(df)

In [35]:
### TEST
test_func(handle_missing_values_in_monthly_visits_and_monthly_active_minutes, "498b490d6c4b8d4f7b372a0f864951417a0da5db128a265f8a9aef41862a6345")

In [36]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_avg_session_minutes(df):
    """
    Impute or remove the missing values from the `avg_session_minutes` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df['avg_session_minutes'] = df['avg_session_minutes'].fillna(df['avg_session_minutes'].median())
    return df
    ### END SOLUTION

In [37]:
df = handle_missing_values_in_avg_session_minutes(df)

In [38]:
### TEST
test_func(handle_missing_values_in_avg_session_minutes, "2fdc6fb34a98f01706a2b77f204d6d54b6369791378f2262c9445ac444ee3c48")

In [39]:
### AUTOMATICALLY GRADED TASK
def handle_missing_values_in_bmi(df):
    """
    Impute or remove the missing values from the `bmi` column.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the dataset as a `pd.DataFrame`.
    """

    ### BEGIN SOLUTION
    df["bmi"] = df["bmi"].fillna(df["bmi"].mode()[0])
    return df
    ### END SOLUTION

In [40]:
df = handle_missing_values_in_bmi(df)

In [41]:
### TEST
test_func(handle_missing_values_in_bmi, "cfe18fa7cd4caf5820a229a092a345e91e6343f8f3fee5c62faa424ac3b641b5")

In [42]:
### AUTOMATICALLY GRADED TASK
def split_dataset_into_train_and_test(df):
    """
    Split the dataset into features `X` and target `y`, where the target is `membership_renewed`.
    Then, divide `X` and `y` into training and test sets using an 85:15 ratio,
    ensuring that the class distribution of y is preserved in both sets.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.
    
    Return `train_X`, `test_X`, `train_y`, and `test_y`.
    """

    ### BEGIN SOLUTION
    X, y = df.drop(columns=["membership_renewed"]), df["membership_renewed"]

    train_X, test_X, train_y, test_y = train_test_split(
        X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
    )

    return train_X, test_X, train_y, test_y
    ### END SOLUTION

In [43]:
train_X, test_X, train_y, test_y = split_dataset_into_train_and_test(df)

In [44]:
### TEST
test_partition(
    split_dataset_into_train_and_test,
    "5d4b6f51aa65a6dc3e1d137a8aa89a76ae34c04498c9cabd04409c950d1efd37",
    "f2e652163952d8dfd5587b3ce542096db484bf840119ba67435ebe11d0b8ab05",
    "0523e73a4f39bc7ed8fdbccac82a33d5897fed196acec9e3cefccb00f9623576",
    "3fd1c9929d2fa8aa46b4922db0c18a3d34783e7868026ea2d8872ddb713ac4fd",
)

In [45]:
### AUTOMATICALLY GRADED TASK
def fit_model(train_X, train_y):
    """
    Fit a boosting model to predict `y` using `X` with 50 estimators, learning rate 0.05 and a maximum depth of 5.

    Use `random_state=RANDOM_STATE` to ensure reproducibility.

    Return the fitted model.
    """

    ### BEGIN SOLUTION
    return XGBClassifier(    
        n_estimators=50,
        learning_rate=0.05,
        max_depth=5,
        random_state = RANDOM_STATE).fit(train_X, train_y)
    ### END SOLUTION    

In [46]:
model = fit_model(train_X, train_y)
pred_y = model.predict(test_X)

In [47]:
### TEST
assert isinstance(model, XGBClassifier)
params = model.get_params()
assert params["n_estimators"] == 50
assert params["max_depth"] == 5
assert abs(params["learning_rate"] - 0.05) < 1e-12
assert params["random_state"] == RANDOM_STATE
assert pred_y.shape[0] == test_X.shape[0]

In [48]:
def evaluate_model(test_y, pred_y):
    """
    Evaluate the model using precision, recall, and F1-score, with a weighted average.

    Return `precision`, `recall`, and `f1` rounded with 2 decimals.
    """
    
    ### BEGIN SOLUTION
    precision = precision_score(test_y, pred_y, average="weighted")
    recall = recall_score(test_y, pred_y, average="weighted")
    f1 = f1_score(test_y, pred_y, average="weighted")

    return round(precision, 2), round(recall, 2), round(f1, 2)
    ### END SOLUTION


In [49]:
precision, recall, f1 = evaluate_model(test_y, pred_y)

In [50]:
### TEST
assert precision > 0.70
assert recall > 0.70
assert f1 > 0.70